In [1]:
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import os

In [2]:
BASE_DIR = Path(os.getcwd()).resolve().parent
CSV_DATA_PATH = BASE_DIR / "data"
ASP_PROGRAM_VESSEL = BASE_DIR / "src" / "asp_code"

In [ ]:
vessel_position_dataset_path = CSV_DATA_PATH / "nari_dynamic.csv"
vessel_position_dataset = pd.read_csv(vessel_position_dataset_path, sep=",")

vessel_info_dataset_path = CSV_DATA_PATH / "nari_static.csv"
vessel_info_dataset = pd.read_csv(vessel_info_dataset_path, sep=",")

In [4]:
print(vessel_info_dataset)

filter = (vessel_info_dataset['sourcemmsi'] == 982571400)
vessel_info_dataset.drop(vessel_info_dataset.loc[filter].index, inplace=True)

print(vessel_info_dataset)

         sourcemmsi  imonumber  callsign           shipname  shiptype  tobow  \
0         304091000  9509255.0    V2GU5     HC JETTE-MARIT       70.0  130.0   
1         228037600        0.0     FIHX    AEROUANT BREIZH       30.0    6.0   
2         228064900  8304816.0     FITO          VN SAPEUR       51.0   21.0   
3         227705102   262144.0  FGD5860              BINDY       60.0    9.0   
4         227415000        0.0     FHAF   F/V JEREMI SIMON       90.0   11.0   
...             ...        ...       ...                ...       ...    ...   
1078612   982571400        NaN       NaN   NORMAND JARL MOB      59.0    1.0   
1078613   982571400        NaN       NaN   NORMAND JARL MOB      59.0    1.0   
1078614   982571400        NaN       NaN   NORMAND JARL MOB      59.0    1.0   
1078615   982571400        NaN       NaN   NORMAND JARL MOB      59.0    1.0   
1078616   982571400        NaN       NaN   NORMAND JARL MOB      59.0    1.0   

         tostern  tostarboard  toport  

Computing coordinates, boxes and so on in meters.

In [5]:
def compute_box_margin_m(to_bow, to_stern, to_port, to_starboard):
    """
    Computes half-dimensions of the vessel bounding box in meters,
    based solely on the physical dimensions reported in AIS static data.
    """
    half_length_m = (to_bow + to_stern) / 2.0
    half_width_m  = (to_port + to_starboard) / 2.0
    return half_length_m, half_width_m

vessel_dimensions_m = {}
for _, row in vessel_info_dataset.iterrows():
    if row['sourcemmsi'] in vessel_dimensions_m:
        continue
    halfY_m, halfX_m = compute_box_margin_m(
        row['tobow'], row['tostern'], row['toport'], row['tostarboard'])
    vessel_dimensions_m[row['sourcemmsi']] = (halfX_m, halfY_m)

In [6]:
import math

R = 6371000.0  # Earth radius in meters
LAT0_DEG = 48.0
LON0_DEG = -5.0
LAT0 = math.radians(LAT0_DEG)
LON0 = math.radians(LON0_DEG)

def latlon_to_xy_m(lat_deg, lon_deg):
    lat = math.radians(lat_deg)
    lon = math.radians(lon_deg)
    x = R * (lon - LON0) * math.cos(LAT0)
    y = R * (lat - LAT0)
    return int(x), int(y)

In [7]:
def assemble_predicate(vessel_id, x_m, y_m, time):
    return f'vessel({vessel_id}, {x_m}, {y_m}, {time}).'

def assemble_vessel_dimension_info(vessel_id):
    try:
        halfX_m, halfY_m = vessel_dimensions_m[vessel_id]
        halfX_m, halfY_m = int(halfX_m), int(halfY_m)
    except KeyError:
        raise KeyError("vessel_id not in vessel_dimensions_m.")

    return f'halfLon({vessel_id}, {halfX_m}). halfLat({vessel_id}, {halfY_m}).'

In [8]:
vessel_info_path = ASP_PROGRAM_VESSEL / "vessels_info_int.lp"
visited_vessels = []

def save_vessel_info(row):
    with open(vessel_info_path, "a") as f:
        vessel_id = int(row['sourcemmsi'])
        time = int(row["t"])
        lat = row['lat']
        lon = row['lon']

        if vessel_id not in vessel_dimensions_m:
            return False

        x_m, y_m = latlon_to_xy_m(lat, lon)

        f.writelines(assemble_predicate(vessel_id, x_m, y_m, time) + "\n")
        if vessel_id not in visited_vessels:
            visited_vessels.append(vessel_id)
            f.writelines(assemble_vessel_dimension_info(vessel_id) + "\n")
        return True


for i, (time, vessels_for_time) in enumerate(vessel_position_dataset.groupby("t", sort=False)):
    if i >= 1000:
        break
    for _, row in vessels_for_time.iterrows():
        save_vessel_info(row)